# RMSA-Net revision experiments, v2 (self-contained)
**What changed from v1:** text length now comes from the BERT attention mask (v1 let PSG and CMTA attend to padding); PSG tokens are mapped to words; frame-level 50% missing test; MMR-rec on unaligned data; official TFR-Net baseline.

Results go to a **new** folder `rmsa_results_v2`; v1 results are superseded.

Run **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom. After a disconnect, use **Runtime → Run all**: finished work is skipped and interrupted runs resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install vaderSentiment tabulate
import torch, os
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime to T4')
os.makedirs('/content/rmsa_code', exist_ok=True)

## 1. Write the code (just run these three cells)

In [ ]:
%%writefile /content/rmsa_code/rmsa_net.py
"""
RMSA-Net: re-implementation of the architecture described in Section III of the
IEEE Access manuscript, plus every experiment requested by the reviewers.

One file: data loading, model (Eqs. 1-28), baselines (TFN, LF-LSTM, MulT-compact),
training (Eqs. 29-31), the experiment registry, and analysis hooks.

Usage (Colab):
    python rmsa_net.py --mosi_aligned /content/drive/MyDrive/MOSI/aligned_50.pkl \
                       --mosi_unaligned /content/drive/MyDrive/MOSI/unaligned_50.pkl \
                       --groups main ablation mmr_analysis unaligned baselines \
                       --out /content/drive/MyDrive/rmsa_results
    python analyze.py --out /content/drive/MyDrive/rmsa_results

Smoke test on synthetic data (CPU, ~1 minute):
    python rmsa_net.py --smoke --out /tmp/rmsa_smoke
"""
from __future__ import annotations

import argparse, json, math, os, pickle, time, random
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =====================================================================
# 1. Reproducibility
# =====================================================================
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =====================================================================
# 2. Data
# =====================================================================
def _first(d: dict, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None


def _compact(x: np.ndarray, lengths: Optional[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    """Return sequences with valid (non-zero) frames moved to the front, plus lengths.
    If the file provides lengths (MMSA format, padding at the end) they are used as-is."""
    x = np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if lengths is not None:
        return x, np.clip(np.asarray(lengths).astype(np.int64), 1, x.shape[1])
    valid = np.abs(x).sum(-1) > 0                                # (N, T)
    order = np.argsort(~valid, axis=1, kind="stable")             # valid rows first
    x = np.take_along_axis(x, order[..., None], axis=1)
    return x, np.clip(valid.sum(1), 1, x.shape[1])


def load_split_file(path: str) -> Dict[str, dict]:
    """Loads CMU-MOSI / MOSEI pickles in either the MMSA format
    (keys: text/audio/vision/regression_labels[/audio_lengths/vision_lengths/raw_text])
    or the MulT format (keys: text/audio/vision/labels)."""
    with open(path, "rb") as f:
        data = pickle.load(f)
    out = {}
    for split, alias in [("train", ["train"]), ("valid", ["valid", "dev", "val"]), ("test", ["test"])]:
        raw = _first(data, alias)
        if raw is None:
            raise KeyError(f"{path}: no split {alias}")
        text = _first(raw, ["text", "glove", "text_glove"])
        audio = _first(raw, ["audio", "covarep"])
        vision = _first(raw, ["vision", "facet", "openface"])
        labels = _first(raw, ["regression_labels", "labels"])
        if text is None or audio is None or vision is None or labels is None:
            raise KeyError(f"{path}/{split}: missing text/audio/vision/labels; keys = {list(raw.keys())}")
        if text.ndim != 3:
            raise ValueError(f"{path}: text must be (N,T,d); got {text.shape}. "
                             "Use a release with GloVe (300-d) or BERT feature sequences, not token ids.")
        # MMSA releases carry BERT token ids + attention mask in text_bert (N, 3, T). Use the mask for the true
        # text length: BERT features at [PAD] positions are non-zero, so zero-row detection would count padding.
        tb = _first(raw, ["text_bert"])
        ids, tl_given = None, None
        if tb is not None and np.asarray(tb).ndim == 3 and np.asarray(tb).shape[1] == 3 \
                and np.asarray(tb).shape[2] == np.asarray(text).shape[1]:
            tb = np.asarray(tb)
            ids, tl_given = tb[:, 0, :].astype(np.int64), tb[:, 1, :].sum(1)
        t, tl = _compact(np.asarray(text), tl_given)
        a, al = _compact(np.asarray(audio), _first(raw, ["audio_lengths"]))
        v, vl = _compact(np.asarray(vision), _first(raw, ["vision_lengths"]))
        y = np.clip(np.asarray(labels, dtype=np.float32).reshape(-1), -3, 3)
        out[split] = dict(text=t, audio=a, vision=v, tl=tl, al=al, vl=vl, y=y, text_ids=ids,
                          raw_text=_first(raw, ["raw_text"]), ids=_first(raw, ["id", "ids"]))
    # Count segments where a modality is entirely missing in the file itself (R1.7: report this)
    out["_stats"] = {s: {"n": int(len(out[s]["y"])),
                         "audio_all_zero": int((np.abs(out[s]["audio"]).sum((1, 2)) == 0).sum()),
                         "vision_all_zero": int((np.abs(out[s]["vision"]).sum((1, 2)) == 0).sum()),
                         "T_text": int(out[s]["text"].shape[1]), "text_len_mean": float(out[s]["tl"].mean()),
                         "text_len_source": "text_bert attention mask" if out[s]["text_ids"] is not None else "non-zero rows", "T_audio": int(out[s]["audio"].shape[1]),
                         "T_vision": int(out[s]["vision"].shape[1]),
                         "d": [int(out[s]["text"].shape[2]), int(out[s]["audio"].shape[2]), int(out[s]["vision"].shape[2])]}
                     for s in ["train", "valid", "test"]}
    return out


def standardize(ds: Dict[str, dict]):
    """z-score audio and vision with training statistics over valid frames (text untouched)."""
    for m, lk in [("audio", "al"), ("vision", "vl")]:
        x, l = ds["train"][m], ds["train"][lk]
        mask = np.arange(x.shape[1])[None, :] < l[:, None]
        frames = x[mask]
        frames = frames[np.abs(frames).sum(-1) > 0] if len(frames) else frames
        mu, sd = frames.mean(0), frames.std(0) + 1e-6
        for s in ["train", "valid", "test"]:
            X = ds[s][m]; L = ds[s][lk]
            msk = (np.arange(X.shape[1])[None, :] < L[:, None]) & (np.abs(X).sum(-1) > 0)
            X[msk] = (X[msk] - mu) / sd
        ds["_std_" + m] = float(sd.mean())


def synthetic_dataset(n=(300, 60, 120), T=(20, 60, 40), d=(300, 5, 20), seed=0) -> Dict[str, dict]:
    """Small synthetic data set with a learnable signal; used only for the smoke test."""
    rng = np.random.RandomState(seed)
    out = {}
    w = [rng.randn(dd) / math.sqrt(dd) for dd in d]
    for split, N in zip(["train", "valid", "test"], n):
        y = np.clip(rng.randn(N) * 1.5, -3, 3).astype(np.float32)
        feats, lens = [], []
        for k, (TT, dd) in enumerate(zip(T, d)):
            L = rng.randint(TT // 2, TT + 1, size=N)
            x = rng.randn(N, TT, dd).astype(np.float32) * 0.5
            x += (y[:, None, None] * w[k][None, None, :] * (1.5 if k == 0 else 0.6)).astype(np.float32)
            x[np.arange(TT)[None, :] >= L[:, None]] = 0
            feats.append(x); lens.append(L)
        vocab = ["good", "bad", "great", "terrible", "movie", "the", "plot", "was", "really", "boring", "love", "it"]
        raw = np.array([" ".join(rng.choice(vocab, int(L))) for L in lens[0]])
        out[split] = dict(text=feats[0], audio=feats[1], vision=feats[2], tl=lens[0], al=lens[1], vl=lens[2],
                          y=y, raw_text=raw, ids=None)
    out["_stats"] = {}
    return out


@dataclass
class Degrade:
    """A test (or training) degradation condition."""
    name: str = "clean"
    pa: float = 0.0          # P(audio stream entirely missing)
    pv: float = 0.0          # P(vision stream entirely missing)
    frame_drop: float = 0.0  # fraction of frames zeroed inside present a/v streams (partial loss)
    noise: float = 0.0       # Gaussian noise std (in standardised feature units) added to present a/v streams


class Batcher:
    """Yields padded mini-batches with missing-modality simulation.

    Training masks are re-drawn every epoch (Eq. 29). Evaluation masks are fixed per seed:
    uniforms U_a, U_v ~ U(0,1) are drawn once from mask_seed and mu_m = 1[U_m < p_m], so the
    masks for different rates are nested and identical across all conditions of one seed
    (paired comparisons see the same test mask)."""

    def __init__(self, split: dict, batch_size: int, mask_seed: int):
        self.s = split
        self.N = len(split["y"])
        self.bs = batch_size
        r = np.random.RandomState(mask_seed)
        self.Ua, self.Uv = r.rand(self.N), r.rand(self.N)
        self.frame_rng_seed = mask_seed + 7

    def _apply(self, idx, deg: Degrade, train: bool, gen: Optional[np.random.RandomState]):
        s = self.s
        text = torch.from_numpy(s["text"][idx]); audio = torch.from_numpy(s["audio"][idx].copy())
        vision = torch.from_numpy(s["vision"][idx].copy())
        tl = torch.from_numpy(s["tl"][idx]); al = torch.from_numpy(s["al"][idx].copy()); vl = torch.from_numpy(s["vl"][idx].copy())
        if train:
            mu_a = torch.from_numpy((gen.rand(len(idx)) < deg.pa).astype(np.float32))
            mu_v = torch.from_numpy((gen.rand(len(idx)) < deg.pv).astype(np.float32))
        else:
            mu_a = torch.from_numpy((self.Ua[idx] < deg.pa).astype(np.float32))
            mu_v = torch.from_numpy((self.Uv[idx] < deg.pv).astype(np.float32))
        frng = gen if train else np.random.RandomState(self.frame_rng_seed + int(idx[0]))
        for x, L, mu in [(audio, al, mu_a), (vision, vl, mu_v)]:
            if deg.frame_drop > 0:
                drop = torch.from_numpy(frng.rand(*x.shape[:2]) < deg.frame_drop)
                x[drop] = 0.0
            if deg.noise > 0:
                valid = (torch.arange(x.shape[1])[None, :] < L[:, None]).float()[..., None]
                x += torch.from_numpy(frng.randn(*x.shape).astype(np.float32)) * deg.noise * valid
            x[mu.bool()] = 0.0                       # whole stream missing -> all-zero tensor
            L[mu.bool()] = 1
        return dict(text=text, audio=audio, vision=vision, tl=tl, al=al, vl=vl,
                    y=torch.from_numpy(s["y"][idx]), idx=torch.from_numpy(idx))

    def iterate(self, deg: Degrade, train: bool, gen: Optional[np.random.RandomState] = None):
        order = gen.permutation(self.N) if train else np.arange(self.N)
        for i in range(0, self.N, self.bs):
            idx = order[i:i + self.bs]
            b = self._apply(idx, deg, train, gen)
            yield {k: v.to(DEVICE) for k, v in b.items()}


# =====================================================================
# 3. Model (Section III)
# =====================================================================
def lengths_mask(lengths: torch.Tensor, T: int) -> torch.Tensor:
    return torch.arange(T, device=lengths.device)[None, :] < lengths[:, None]


def masked_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m = mask.float().unsqueeze(-1)
    return (x * m).sum(1) / m.sum(1).clamp(min=1.0)


class BiLSTMEncoder(nn.Module):
    """Eqs. (1)-(3). hidden = per-direction size; output dim = 2*hidden."""
    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, num_layers=layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)

    def forward(self, x, lengths):
        T = x.shape[1]
        packed = pack_padded_sequence(x, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=T)
        return out


class PSG(nn.Module):
    """Polarity Sentiment Gate, Eqs. (13)-(17).
    G = sigma(Wg2 tanh(Wg1 X + b) + b);  X~ = X * G;  p = tanh(Wp2 ReLU(Wp1 X~ + b) + b);
    alpha = softmax(|p|) over valid tokens;  c_t = sum_i alpha_i X~_i."""
    def __init__(self, d_t, H):
        super().__init__()
        self.g1, self.g2 = nn.Linear(d_t, H), nn.Linear(H, d_t)
        self.p1, self.p2 = nn.Linear(d_t, H), nn.Linear(H, 1)

    def forward(self, X, tmask):
        G = torch.sigmoid(self.g2(torch.tanh(self.g1(X))))
        Xg = X * G
        p = torch.tanh(self.p2(F.relu(self.p1(Xg)))).squeeze(-1)          # (B,T)
        score = p.abs().masked_fill(~tmask, float("-inf"))
        alpha = torch.softmax(score, dim=1)
        c = (alpha.unsqueeze(-1) * Xg).sum(1)
        return Xg, c, alpha, p


class CMTA(nn.Module):
    """Cross-Modal Temporal Attention, Eqs. (7)-(12): single head, text queries,
    audio/visual keys/values at native length, padding masked, mean-pooled over valid queries."""
    def __init__(self, d_q, d_k, H):
        super().__init__()
        self.Wq, self.Wk, self.Wv = nn.Linear(d_q, H, bias=False), nn.Linear(d_k, H, bias=False), nn.Linear(d_k, H, bias=False)
        self.H = H

    def forward(self, Hq, qmask, Hk, kmask):
        Q, K, V = self.Wq(Hq), self.Wk(Hk), self.Wv(Hk)
        s = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(self.H)
        s = s.masked_fill(~kmask[:, None, :], float("-inf"))
        A = torch.softmax(s, dim=-1)
        out = torch.bmm(A, V)
        return masked_mean(out, qmask), A


class RMSANet(nn.Module):
    """mmr_mode: 'mmr'     hard-switch imputation, Eq. (6)
                 'zero'    zero-fill: absent projection is used as-is (no surrogate)
                 'textmlp' control: zero-fill + an always-on text MLP branch R_m(f_t) concatenated to the fusion input
                           (same extra parameters as MMR, no switch) -- tests whether MMR gains are just extra text capacity."""
    def __init__(self, d_t, d_a, d_v, H=128, D=64, dropout=0.3,
                 use_psg=True, use_cmta=True, mmr_mode="mmr"):
        super().__init__()
        self.use_psg, self.use_cmta, self.mmr_mode = use_psg, use_cmta, mmr_mode
        self.psg = PSG(d_t, H) if use_psg else None
        self.enc_t = BiLSTMEncoder(d_t, H, 2, dropout)              # -> 2H = 256
        self.enc_a = BiLSTMEncoder(d_a, H // 2, 1, 0.0)             # -> H = 128
        self.enc_v = BiLSTMEncoder(d_v, H // 2, 1, 0.0)
        if use_cmta:
            self.cmta_a, self.cmta_v = CMTA(2 * H, H, H), CMTA(2 * H, H, H)
        self.proj_t = nn.Linear(2 * H + d_t, D)                      # Eq. (18), W_t in R^{D x 556}
        self.proj_a = nn.Linear(2 * H, D)                            # Eq. (19)
        self.proj_v = nn.Linear(2 * H, D)                            # Eq. (20)
        self.R_a = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, D))   # Eq. (5)
        self.R_v = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, D))
        n_in = 5 * D if mmr_mode == "textmlp" else 3 * D
        self.fuse = nn.Sequential(nn.Linear(n_in, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, D // 2), nn.ReLU())  # Eq. (21)
        self.head_reg, self.head_cls = nn.Linear(D // 2, 1), nn.Linear(D // 2, 1)          # Eqs. (22)-(23)
        self.uni = nn.ModuleDict({m: nn.Linear(D, 1) for m in "tav"})                       # Eq. (24)

    def forward(self, b):
        X, Xa, Xv = b["text"], b["audio"], b["vision"]
        tmask = lengths_mask(b["tl"], X.shape[1])
        amask = lengths_mask(b["al"], Xa.shape[1])
        vmask = lengths_mask(b["vl"], Xv.shape[1])
        # Eq. (4): absence indicator from the input itself
        mu_a = (Xa.abs().sum((1, 2)) == 0).float()
        mu_v = (Xv.abs().sum((1, 2)) == 0).float()

        if self.use_psg:
            Xg, c_t, alpha, p = self.psg(X, tmask)
        else:
            Xg, c_t, alpha, p = X, masked_mean(X, tmask), None, None
        Ht = self.enc_t(Xg, b["tl"]); Ha = self.enc_a(Xa, b["al"]); Hv = self.enc_v(Xv, b["vl"])
        Ht_bar, Ha_bar, Hv_bar = masked_mean(Ht, tmask), masked_mean(Ha, amask), masked_mean(Hv, vmask)
        if self.use_cmta:
            h_ta, A_ta = self.cmta_a(Ht, tmask, Ha, amask)
            h_tv, A_tv = self.cmta_v(Ht, tmask, Hv, vmask)
        else:
            h_ta, h_tv, A_ta, A_tv = Ha_bar, Hv_bar, None, None
        f_t = F.relu(self.proj_t(torch.cat([Ht_bar, c_t], -1)))
        f_a = F.relu(self.proj_a(torch.cat([Ha_bar, h_ta], -1)))
        f_v = F.relu(self.proj_v(torch.cat([Hv_bar, h_tv], -1)))
        fh_a, fh_v = self.R_a(f_t), self.R_v(f_t)

        if self.mmr_mode == "mmr":                                     # Eq. (6)
            ft_a = (1 - mu_a)[:, None] * f_a + mu_a[:, None] * fh_a
            ft_v = (1 - mu_v)[:, None] * f_v + mu_v[:, None] * fh_v
            fused_in = torch.cat([f_t, ft_a, ft_v], -1)
        elif self.mmr_mode == "zero":
            fused_in = torch.cat([f_t, f_a, f_v], -1)
        elif self.mmr_mode == "textmlp":
            fused_in = torch.cat([f_t, f_a, f_v, fh_a, fh_v], -1)
        else:
            raise ValueError(self.mmr_mode)
        z = self.fuse(fused_in)
        return dict(y_reg=self.head_reg(z).squeeze(-1), y_cls=self.head_cls(z).squeeze(-1),
                    y_uni={m: self.uni[m](f).squeeze(-1) for m, f in zip("tav", [f_t, f_a, f_v])},
                    f_t=f_t, f_a=f_a, f_v=f_v, fh_a=fh_a, fh_v=fh_v, mu_a=mu_a, mu_v=mu_v,
                    A_ta=A_ta, A_tv=A_tv, alpha=alpha, p=p, tmask=tmask)


# ---------------------------------------------------------------------
# Baselines, trained in the same pipeline (same features, masks, seeds)
# ---------------------------------------------------------------------
class LFLSTM(nn.Module):
    """Late-fusion BiLSTM: per-modality BiLSTM, mean-pool, concat, MLP. Zero-fill for missing input."""
    def __init__(self, d_t, d_a, d_v, H=128, dropout=0.3):
        super().__init__()
        self.e = nn.ModuleList([BiLSTMEncoder(d, H // 2, 1, 0.0) for d in (d_t, d_a, d_v)])
        self.mlp = nn.Sequential(nn.Linear(3 * H, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))

    def forward(self, b):
        hs = []
        for enc, x, l in zip(self.e, (b["text"], b["audio"], b["vision"]), (b["tl"], b["al"], b["vl"])):
            hs.append(masked_mean(enc(x, l), lengths_mask(l, x.shape[1])))
        return dict(y_reg=self.mlp(torch.cat(hs, -1)).squeeze(-1))


class TFN(nn.Module):
    """Tensor Fusion Network (Zadeh et al., 2017): text LSTM subnet, mean-pooled audio/visual DNN subnets,
    3-fold outer product with appended 1s, post-fusion MLP."""
    def __init__(self, d_t, d_a, d_v, ht=64, ha=16, hv=16, post=128, dropout=0.3):
        super().__init__()
        self.text = BiLSTMEncoder(d_t, ht // 2, 1, 0.0)
        self.audio = nn.Sequential(nn.Linear(d_a, ha), nn.ReLU(), nn.Linear(ha, ha), nn.ReLU())
        self.vision = nn.Sequential(nn.Linear(d_v, hv), nn.ReLU(), nn.Linear(hv, hv), nn.ReLU())
        self.post = nn.Sequential(nn.Dropout(dropout), nn.Linear((ht + 1) * (ha + 1) * (hv + 1), post), nn.ReLU(),
                                  nn.Linear(post, post), nn.ReLU(), nn.Linear(post, 1))

    def forward(self, b):
        zt = masked_mean(self.text(b["text"], b["tl"]), lengths_mask(b["tl"], b["text"].shape[1]))
        za = self.audio(masked_mean(b["audio"], lengths_mask(b["al"], b["audio"].shape[1])))
        zv = self.vision(masked_mean(b["vision"], lengths_mask(b["vl"], b["vision"].shape[1])))
        one = torch.ones(zt.shape[0], 1, device=zt.device)
        zt, za, zv = torch.cat([zt, one], 1), torch.cat([za, one], 1), torch.cat([zv, one], 1)
        fusion = torch.einsum("bi,bj,bk->bijk", zt, za, zv).flatten(1)
        return dict(y_reg=self.post(fusion).squeeze(-1))


def sinusoid(T, d, device):
    pos = torch.arange(T, device=device).float()[:, None]
    i = torch.arange(d, device=device).float()[None, :]
    ang = pos / torch.pow(10000, (2 * (i // 2)) / d)
    return torch.where(i % 2 == 0, torch.sin(ang), torch.cos(ang))


class XBlock(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.ln_q, self.ln_k, self.ln_f = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.att = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4 * d, d))

    def forward(self, q, k, kpad):
        kk = self.ln_k(k)
        q = q + self.att(self.ln_q(q), kk, kk, key_padding_mask=kpad, need_weights=False)[0]
        return q + self.ff(self.ln_f(q))


class MulTCompact(nn.Module):
    """Compact re-implementation of MulT (Tsai et al., 2019): 1-D conv projections, six directed
    cross-modal transformers, per-target self-attention, residual MLP head. Masked mean-pooling replaces
    'last time step' so that variable-length padded sequences are handled. Defaults follow MulT's
    CMU-MOSI setting (d = 30, 5 heads, 4 layers)."""
    def __init__(self, d_t, d_a, d_v, d=30, heads=5, layers=4, dropout=0.1):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv1d(di, d, 1, bias=False) for di in (d_t, d_a, d_v)])
        self.cross = nn.ModuleDict({f"{t}{s}": nn.ModuleList([XBlock(d, heads, dropout) for _ in range(layers)])
                                    for t in "tav" for s in "tav" if t != s})
        self.mem = nn.ModuleDict({t: nn.ModuleList([XBlock(2 * d, heads, dropout) for _ in range(3)]) for t in "tav"})
        self.out = nn.Sequential(nn.Linear(6 * d, 6 * d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(6 * d, 6 * d))
        self.head = nn.Linear(6 * d, 1)

    def forward(self, b):
        xs = {m: self.proj[i](b[k].transpose(1, 2)).transpose(1, 2) for i, (m, k) in enumerate(zip("tav", ("text", "audio", "vision")))}
        L = {"t": b["tl"], "a": b["al"], "v": b["vl"]}
        masks = {m: lengths_mask(L[m], xs[m].shape[1]) for m in "tav"}
        xs = {m: x + sinusoid(x.shape[1], x.shape[2], x.device)[None] for m, x in xs.items()}
        hs = []
        for t in "tav":
            outs = []
            for s in "tav":
                if s == t:
                    continue
                h = xs[t]
                for blk in self.cross[f"{t}{s}"]:
                    h = blk(h, xs[s], ~masks[s])
                outs.append(h)
            h = torch.cat(outs, -1)
            for blk in self.mem[t]:
                h = blk(h, h, ~masks[t])
            hs.append(masked_mean(h, masks[t]))
        h = torch.cat(hs, -1)
        return dict(y_reg=self.head(h + self.out(h)).squeeze(-1))


# =====================================================================
# 4. Loss (Eqs. 25-28 and the optional feature-matching term 25b)
# =====================================================================
def rmsa_loss(out, y, lam1=0.5, lam2=0.1, lam3=0.0, mask_uni=True):
    L = F.mse_loss(out["y_reg"], y)
    if "y_cls" not in out:
        return L, {}
    L_cls = F.binary_cross_entropy_with_logits(out["y_cls"], (y >= 0).float())
    L_uni = 0.0
    for m in "tav":
        e = (out["y_uni"][m] - y) ** 2
        if mask_uni and m != "t":
            keep = 1 - out[f"mu_{m}"]
            e = (e * keep).sum() / keep.sum().clamp(min=1)
        else:
            e = e.mean()
        L_uni = L_uni + e
    L = L + lam1 * L_cls + lam2 * L_uni
    L_rec = torch.tensor(0.0, device=y.device)
    if lam3 > 0:
        for m in "av":
            keep = 1 - out[f"mu_{m}"]
            se = ((out[f"fh_{m}"] - out[f"f_{m}"].detach()) ** 2).sum(-1)       # stop-gradient on target
            L_rec = L_rec + (se * keep).sum() / keep.sum().clamp(min=1)
        L = L + lam3 * L_rec
    return L, {"rec": float(L_rec.detach())}


# =====================================================================
# 5. Metrics
# =====================================================================
def metrics(pred: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    from sklearn.metrics import f1_score
    pred = np.clip(pred, -3, 3)
    mae = float(np.abs(pred - y).mean())
    corr = float(np.corrcoef(pred, y)[0, 1]) if np.std(pred) > 1e-8 else 0.0
    acc7 = float((np.clip(np.round(pred), -3, 3) == np.clip(np.round(y), -3, 3)).mean() * 100)
    pb, yb = pred >= 0, y >= 0                                       # paper convention (Eq. 34)
    acc2 = float((pb == yb).mean() * 100)
    f1 = float(f1_score(yb, pb, average="weighted") * 100)
    nz = y != 0                                                      # common 'non-zero' convention
    acc2_nz = float(((pred[nz] > 0) == (y[nz] > 0)).mean() * 100)
    f1_nz = float(f1_score(y[nz] > 0, pred[nz] > 0, average="weighted") * 100)
    return dict(mae=mae, corr=corr, acc7=acc7, acc2=acc2, f1=f1, acc2_nz=acc2_nz, f1_nz=f1_nz)


@torch.no_grad()
def evaluate(model, batcher: Batcher, deg: Degrade, extras=False) -> Dict[str, float]:
    model.eval()
    preds, ys = [], []
    cos = {"a": [], "v": []}; mse = {"a": [], "v": []}
    ent = {"a": [], "v": []}
    for b in batcher.iterate(deg, train=False):
        out = model(b)
        preds.append(out["y_reg"].cpu().numpy()); ys.append(b["y"].cpu().numpy())
        if extras and "fh_a" in out:
            for m in "av":
                pres = out[f"mu_{m}"] == 0
                if pres.any():
                    cos[m].append(F.cosine_similarity(out[f"fh_{m}"][pres], out[f"f_{m}"][pres], dim=-1).cpu().numpy())
                    mse[m].append(((out[f"fh_{m}"][pres] - out[f"f_{m}"][pres]) ** 2).sum(-1).cpu().numpy())
                A = out.get(f"A_t{m}")
                if A is not None and pres.any():
                    L = b[f"{m}l"][pres].float()
                    Ap = A[pres].clamp(min=1e-12)
                    H = -(Ap * Ap.log()).sum(-1)                        # (B',Tq)
                    qm = out["tmask"][pres].float()
                    Hn = (H / L.log().clamp(min=1e-6)[:, None] * qm).sum(1) / qm.sum(1)   # entropy / log(T_k): 1 = uniform
                    ent[m].append(Hn.cpu().numpy())
    r = metrics(np.concatenate(preds), np.concatenate(ys))
    if extras:
        for m in "av":
            r[f"fid_cos_{m}"] = float(np.concatenate(cos[m]).mean()) if cos[m] else float("nan")
            r[f"fid_mse_{m}"] = float(np.concatenate(mse[m]).mean()) if mse[m] else float("nan")
            r[f"attn_entropy_{m}"] = float(np.concatenate(ent[m]).mean()) if ent[m] else float("nan")
    return r


# =====================================================================
# 6. Experiment configuration and registry
# =====================================================================
@dataclass
class RunCfg:
    dataset: str = "mosi"            # key into the loaded data sets
    model: str = "rmsa"              # rmsa | tfn | lflstm | mult
    use_psg: bool = True
    use_cmta: bool = True
    mmr_mode: str = "mmr"            # mmr | zero | textmlp
    lam3: float = 0.0                # 0 = MMR-task (paper); >0 = MMR-rec
    train: Degrade = field(default_factory=Degrade)
    group: str = ""

    @property
    def name(self) -> str:
        tr = f"tr{self.train.pa:g}-{self.train.pv:g}"
        if self.model != "rmsa":
            return f"{self.dataset}|{self.model}|{tr}"
        flags = ("P" if self.use_psg else "-") + ("C" if self.use_cmta else "-") + {"mmr": "M", "zero": "Z", "textmlp": "X"}[self.mmr_mode]
        rec = f"|rec{self.lam3:g}" if self.lam3 > 0 else ""
        return f"{self.dataset}|rmsa-{flags}{rec}|{tr}"


# Test conditions evaluated for EVERY trained model (inference only, cheap).
TESTS = [Degrade("clean"), Degrade("A30", 0.3, 0), Degrade("V30", 0, 0.3), Degrade("AV30", 0.3, 0.3),
         Degrade("AV50", 0.5, 0.5), Degrade("frame30", frame_drop=0.3), Degrade("frame50", frame_drop=0.5), Degrade("noise0.5", noise=0.5),
         Degrade("noise1.0", noise=1.0)]
TR = {"clean": Degrade("clean"), "A30": Degrade("A30", 0.3, 0), "V30": Degrade("V30", 0, 0.3),
      "AV30": Degrade("AV30", 0.3, 0.3), "AV50": Degrade("AV50", 0.5, 0.5)}


def registry(groups: List[str]) -> List[RunCfg]:
    R: List[RunCfg] = []
    if "main" in groups:              # Table 3 (aligned) and Table 7 (cross-evaluation comes for free)
        R.append(RunCfg(train=TR["clean"], group="main"))
        for c in ["A30", "V30", "AV30", "AV50"]:
            R.append(RunCfg(train=TR[c], group="main"))
            R.append(RunCfg(train=TR[c], mmr_mode="zero", group="main"))
        R.append(RunCfg(train=TR["clean"], mmr_mode="zero", group="main"))
    if "ablation" in groups:          # Table 4: full factorial under A+V 30%
        for psg in (False, True):
            for cmta in (False, True):
                for mm in ("zero", "mmr"):
                    R.append(RunCfg(train=TR["AV30"], use_psg=psg, use_cmta=cmta, mmr_mode=mm, group="ablation"))
    if "mmr_analysis" in groups:      # Table 8: what the surrogate encodes
        R.append(RunCfg(train=TR["AV30"], lam3=0.1, group="mmr_analysis"))
        R.append(RunCfg(train=TR["AV30"], mmr_mode="textmlp", group="mmr_analysis"))
        R.append(RunCfg(train=TR["clean"], lam3=0.1, group="mmr_analysis"))   # MMR-rec can learn R_m even without dropout
        R.append(RunCfg(dataset="mosi_unal", train=TR["clean"], lam3=0.1, group="mmr_analysis"))   # vs TFR-Net (Table 3c)
    if "unaligned" in groups:         # Table 3 unaligned row, Table 6 joint degradation
        R.append(RunCfg(dataset="mosi_unal", train=TR["clean"], group="unaligned"))
        R.append(RunCfg(dataset="mosi_unal", train=TR["clean"], use_cmta=False, group="unaligned"))
        R.append(RunCfg(train=TR["clean"], use_cmta=False, group="unaligned"))            # aligned, no CMTA (Table 10)
        R.append(RunCfg(dataset="mosi_unal", train=TR["AV30"], group="unaligned"))
        R.append(RunCfg(dataset="mosi_unal", train=TR["AV30"], mmr_mode="zero", group="unaligned"))
    if "baselines" in groups:         # Table 3b
        for m in ("lflstm", "tfn", "mult"):
            for c in ("clean", "AV30", "AV50"):
                R.append(RunCfg(model=m, train=TR[c], group="baselines"))
        R.append(RunCfg(dataset="mosi_unal", model="mult", train=TR["clean"], group="baselines"))
        R.append(RunCfg(dataset="mosi_unal", model="mult", train=TR["AV30"], group="baselines"))
    if "mosei" in groups:             # Table 5
        R.append(RunCfg(dataset="mosei", train=TR["clean"], group="mosei"))
        R.append(RunCfg(dataset="mosei", train=TR["AV30"], group="mosei"))
        R.append(RunCfg(dataset="mosei", train=TR["AV30"], mmr_mode="zero", group="mosei"))
    seen, out = set(), []
    for r in R:                        # same config requested by two groups -> run once
        if r.name not in seen:
            seen.add(r.name); out.append(r)
    return out


def build_model(cfg: RunCfg, dims) -> nn.Module:
    d_t, d_a, d_v = dims
    if cfg.model == "rmsa":
        return RMSANet(d_t, d_a, d_v, use_psg=cfg.use_psg, use_cmta=cfg.use_cmta, mmr_mode=cfg.mmr_mode)
    if cfg.model == "tfn":
        return TFN(d_t, d_a, d_v)
    if cfg.model == "lflstm":
        return LFLSTM(d_t, d_a, d_v)
    if cfg.model == "mult":
        return MulTCompact(d_t, d_a, d_v)
    raise ValueError(cfg.model)


# =====================================================================
# 7. Training (Section IV-B)
# =====================================================================
def _save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)          # never leaves a half-written checkpoint (safe if Colab dies mid-save)


def train_one(cfg: RunCfg, data: dict, seed: int, epochs: int, bs: int, lr: float,
              ckpt_dir: Optional[str] = None, tag: str = "run", log=print):
    """Trains one (configuration, seed). If ckpt_dir is given, saves a checkpoint after EVERY epoch
    (model, optimiser, scheduler, best weights so far, learning curve and all RNG states) and resumes
    from it automatically, so an interrupted run continues from its last finished epoch with the
    same random stream as an uninterrupted run."""
    set_seed(seed)
    dims = tuple(data["train"][k].shape[2] for k in ("text", "audio", "vision"))
    model = build_model(cfg, dims).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr if cfg.model != "mult" else 1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=0.0)     # Eq. (30)
    tr = Batcher(data["train"], bs, mask_seed=seed + 1000)
    va = Batcher(data["valid"], bs, mask_seed=seed + 2000)
    gen = np.random.RandomState(seed)
    best, best_state, best_ep, curve, start, spent = float("inf"), None, -1, [], 1, 0.0

    ck = os.path.join(ckpt_dir, f"{tag}_s{seed}.inprogress.pt") if ckpt_dir else None
    if ck and os.path.exists(ck):
        c = torch.load(ck, map_location=DEVICE, weights_only=False)
        if c.get("epochs") == epochs:
            model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
            best, best_ep, curve, spent = c["best"], c["best_ep"], c["curve"], c["spent"]
            best_state = {k: v.to(DEVICE) for k, v in c["best_state"].items()}
            gen.set_state(c["np_gen"]); torch.set_rng_state(c["torch_rng"])
            if torch.cuda.is_available() and c.get("cuda_rng") is not None:
                torch.cuda.set_rng_state_all(c["cuda_rng"])
            start = c["epoch"] + 1
            log(f"    resuming {tag} seed={seed} from epoch {start}/{epochs}")

    t0 = time.time()
    for ep in range(start, epochs + 1):
        model.train()
        for b in tr.iterate(cfg.train, train=True, gen=gen):
            out = model(b)
            loss, _ = rmsa_loss(out, b["y"], lam3=cfg.lam3)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)                       # Eq. (31)
            opt.step()
        sched.step()
        v = evaluate(model, va, cfg.train)            # model selection under the training condition
        curve.append((ep, v["mae"], v["acc2"]))
        if v["mae"] < best:
            best, best_ep = v["mae"], ep
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        if ck:
            _save_atomic(dict(epoch=ep, epochs=epochs, model=model.state_dict(), opt=opt.state_dict(),
                              sched=sched.state_dict(), best=best, best_ep=best_ep, curve=curve,
                              best_state={k: t.cpu() for k, t in best_state.items()},
                              spent=spent + time.time() - t0, np_gen=gen.get_state(), torch_rng=torch.get_rng_state(),
                              cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None), ck)
    model.load_state_dict(best_state)
    n_params = sum(p.numel() for p in model.parameters())
    return model, dict(best_epoch=best_ep, val_mae=best, train_sec=spent + time.time() - t0, n_params=n_params, curve=curve)


def load_trained(cfg: RunCfg, data: dict, seed: int, ckpt_dir: str) -> nn.Module:
    """Reload the best weights of a finished run (for new analyses without retraining)."""
    dims = tuple(data["train"][k].shape[2] for k in ("text", "audio", "vision"))
    model = build_model(cfg, dims).to(DEVICE)
    tag = cfg.name.replace("|", "_")
    model.load_state_dict(torch.load(os.path.join(ckpt_dir, f"{tag}_s{seed}.best.pt"), map_location=DEVICE))
    return model


# =====================================================================
# 8. Analyses for R1.6 / R2.8 (PSG) and figure exports
# =====================================================================
@torch.no_grad()
def psg_sign_flip_check(model: RMSANet, batch) -> float:
    """Negating the last layer of the polarity estimator maps p -> -p; returns the max absolute change
    in the regression output (expected: 0 up to float error). Demonstrates R1.6's point."""
    if not model.use_psg:
        return float("nan")
    model.eval()
    y0 = model(batch)["y_reg"]
    model.psg.p2.weight.mul_(-1); model.psg.p2.bias.mul_(-1)
    y1 = model(batch)["y_reg"]
    model.psg.p2.weight.mul_(-1); model.psg.p2.bias.mul_(-1)
    return float((y0 - y1).abs().max())


@torch.no_grad()
def export_examples(model: RMSANet, data: dict, path: str, n_attn=8):
    """For the whole test set: PSG token weights (alpha, |p|) with words if raw_text exists (for the
    lexicon correlation and qualitative examples, R1.6/R2.8). For the first n_attn utterances: CMTA
    attention matrices (Figure 2, R2.9)."""
    model.eval()
    raw = data["test"].get("raw_text")
    tids = data["test"].get("text_ids")
    ex, attn = [], None
    for b in Batcher(data["test"], 64, mask_seed=0).iterate(Degrade("clean"), train=False):
        out = model(b)
        if attn is None:
            k = min(n_attn, len(b["y"]))
            attn = dict(A_ta=out["A_ta"][:k].cpu().numpy() if out["A_ta"] is not None else np.zeros(0),
                        A_tv=out["A_tv"][:k].cpu().numpy() if out["A_tv"] is not None else np.zeros(0),
                        tl=b["tl"][:k].cpu().numpy(), al=b["al"][:k].cpu().numpy(), vl=b["vl"][:k].cpu().numpy(),
                        index=b["idx"][:k].cpu().numpy())
        for i in range(len(b["y"])):
            idx = int(b["idx"][i]); T = int(b["tl"][i])
            ex.append(dict(index=idx, label=float(b["y"][i]), pred=float(out["y_reg"][i]),
                           words=str(raw[idx]).split() if raw is not None else None,
                           token_ids=tids[idx, :T].tolist() if tids is not None else None,
                           alpha=out["alpha"][i, :T].cpu().tolist() if out["alpha"] is not None else None,
                           abs_p=out["p"][i, :T].abs().cpu().tolist() if out["p"] is not None else None))
    np.savez_compressed(path + "_attn.npz", **attn)
    with open(path + "_psg.json", "w") as f:
        json.dump(ex, f)


def constant_predictor(data: dict) -> Dict[str, float]:
    """R2.3: MAE of always predicting the mean training label (and of predicting 0)."""
    y, mu = data["test"]["y"], float(data["train"]["y"].mean())
    return dict(const_mean_mae=float(np.abs(y - mu).mean()), const_zero_mae=float(np.abs(y).mean()),
                train_label_mean=mu)


# =====================================================================
# 9. Runner
# =====================================================================
def append_rows(csv_path: str, rows: List[dict]):
    import csv
    new = not os.path.exists(csv_path)
    keys = list(rows[0].keys())
    if not new:
        with open(csv_path) as f:
            header = next(csv.reader(f))
        keys = header + [k for k in keys if k not in header]
        if keys != header:      # header grew: rewrite file
            import pandas as pd
            df = pd.read_csv(csv_path)
            df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
            df.to_csv(csv_path, index=False); return
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerows(rows)


def done_runs(csv_path: str) -> set:
    if not os.path.exists(csv_path):
        return set()
    import pandas as pd
    df = pd.read_csv(csv_path)
    return set(zip(df["run"], df["seed"]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mosi_aligned"); ap.add_argument("--mosi_unaligned"); ap.add_argument("--mosei")
    ap.add_argument("--groups", nargs="+", default=["main", "ablation", "mmr_analysis", "unaligned", "baselines"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[13, 42, 7, 17, 99])
    ap.add_argument("--epochs", type=int, default=30); ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--out", required=True)
    ap.add_argument("--no_standardize", action="store_true")
    ap.add_argument("--smoke", action="store_true", help="synthetic data, 2 epochs, 2 seeds")
    ap.add_argument("--analyze_only", action="store_true", help="only rebuild tables/figures from per_run.csv")
    ap.add_argument("--no_keep_best", action="store_true", help="do not keep best weights of finished runs (saves Drive space)")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    if a.analyze_only:
        import analyze; analyze.run(a.out); return
    csv_path = os.path.join(a.out, "per_run.csv")
    ckpt_dir = os.path.join(a.out, "checkpoints"); os.makedirs(ckpt_dir, exist_ok=True)

    data = {}
    if a.smoke:
        data["mosi"] = synthetic_dataset(seed=0)
        data["mosi_unal"] = synthetic_dataset(T=(20, 120, 80), seed=1)
        data["mosei"] = synthetic_dataset(seed=2)
        a.epochs, a.seeds = 2, a.seeds[:2]
    else:
        for key, p in [("mosi", a.mosi_aligned), ("mosi_unal", a.mosi_unaligned), ("mosei", a.mosei)]:
            if p:
                data[key] = load_split_file(p)
    for k, d in data.items():
        if not a.no_standardize:
            standardize(d)
    info = {k: dict(stats=d.get("_stats", {}), **constant_predictor(d)) for k, d in data.items()}
    with open(os.path.join(a.out, "data_info.json"), "w") as f:
        json.dump(info, f, indent=1)
    print(json.dumps(info, indent=1))

    runs = [r for r in registry(a.groups) if r.dataset in data]
    skipped = [r.name for r in registry(a.groups) if r.dataset not in data]
    if skipped:
        print(f"Skipping {len(skipped)} runs whose data set was not given: {sorted(set(s.split('|')[0] for s in skipped))}")
    done = done_runs(csv_path)
    todo = [(r, s) for s in a.seeds for r in runs if (r.name, s) not in done]
    print(f"{len(runs)} configurations x {len(a.seeds)} seeds; {len(todo)} runs to do (resume-safe).")

    for i, (cfg, seed) in enumerate(todo, 1):
        d = data[cfg.dataset]
        tag = cfg.name.replace("|", "_")
        model, tinfo = train_one(cfg, d, seed, a.epochs, a.bs, a.lr, ckpt_dir=ckpt_dir, tag=tag)
        te = Batcher(d["test"], 64, mask_seed=seed + 3000)
        rows = []
        for deg in TESTS:
            m = evaluate(model, te, deg, extras=(cfg.model == "rmsa"))
            rows.append(dict(run=cfg.name, group=cfg.group, dataset=cfg.dataset, model=cfg.model,
                             psg=int(cfg.use_psg), cmta=int(cfg.use_cmta), mmr_mode=cfg.mmr_mode, lam3=cfg.lam3,
                             train_pa=cfg.train.pa, train_pv=cfg.train.pv, seed=seed, test=deg.name,
                             best_epoch=tinfo["best_epoch"], val_mae=tinfo["val_mae"], n_params=tinfo["n_params"],
                             train_sec=round(tinfo["train_sec"], 1), epochs=a.epochs, **m))
        if cfg.model == "rmsa" and cfg.use_psg:
            b = next(Batcher(d["test"], 64, mask_seed=0).iterate(Degrade("clean"), train=False))
            rows[0]["psg_signflip_maxdiff"] = psg_sign_flip_check(model, b)
        if not a.no_keep_best:
            _save_atomic({k: t.cpu() for k, t in model.state_dict().items()}, os.path.join(ckpt_dir, f"{tag}_s{seed}.best.pt"))
        append_rows(csv_path, rows)                    # the run counts as done only after this line
        ip = os.path.join(ckpt_dir, f"{tag}_s{seed}.inprogress.pt")
        if os.path.exists(ip):
            os.remove(ip)
        with open(os.path.join(a.out, f"curve_{tag}_s{seed}.json"), "w") as f:
            json.dump(tinfo["curve"], f)
        if cfg.model == "rmsa" and cfg.use_psg and cfg.use_cmta and seed == a.seeds[0]:
            export_examples(model, d, os.path.join(a.out, f"examples_{tag}_s{seed}"))
        clean = rows[0]
        print(f"[{i}/{len(todo)}] {cfg.name} seed={seed} best_ep={tinfo['best_epoch']} "
              f"clean MAE={clean['mae']:.3f} Acc2={clean['acc2']:.2f} | {tinfo['train_sec']:.0f}s")

    # Build every table, statistic and figure of the paper from per_run.csv (also works on partial results)
    import sys; sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import analyze
    analyze.run(a.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/rmsa_code/analyze.py
"""
Builds every table, statistic and figure for the revised manuscript from results/per_run.csv.

    python analyze.py --out /content/drive/MyDrive/rmsa_results

Writes into --out:
    tables.md      all tables as Markdown (mean ± std over seeds) + paired tests
    tables.tex     the same tables as LaTeX tabular blocks
    stats.csv      every paired comparison: mean difference, 95% CI, t, p, n
    fig3_missing.png, fig4_convergence.png, fig5_ablation.png, fig2_attention.png
    psg_analysis.md   sign-flip check, |p| vs lexicon intensity, qualitative examples
"""
import argparse, glob, json, os
import numpy as np
import pandas as pd
from scipy import stats

M5 = ["mae", "corr", "acc7", "acc2", "f1"]
HDR = {"mae": "MAE↓", "corr": "Corr↑", "acc7": "Acc-7↑", "acc2": "Acc-2↑", "f1": "F1↑",
       "acc2_nz": "Acc-2 (non-0)↑", "f1_nz": "F1 (non-0)↑",
       "fid_cos_a": "cos(f̂a,fa)↑", "fid_cos_v": "cos(f̂v,fv)↑", "fid_mse_a": "MSE(f̂a,fa)↓", "fid_mse_v": "MSE(f̂v,fv)↓",
       "attn_entropy_a": "Attn entropy (audio)", "attn_entropy_v": "Attn entropy (vision)"}
DIG = {"mae": 3, "corr": 3, "acc7": 2, "acc2": 2, "f1": 2, "acc2_nz": 2, "f1_nz": 2}


def fmt(m, s, k):
    d = DIG.get(k, 3)
    if np.isnan(m):
        return "—"
    return f"{m:.{d}f} ± {s:.{d}f}" if not np.isnan(s) else f"{m:.{d}f}"


def cell(df, run, test, k):
    if k not in df.columns:
        return np.nan, np.nan, 0
    x = df[(df.run == run) & (df.test == test)][k].astype(float)
    if len(x) == 0:
        return np.nan, np.nan, 0
    return x.mean(), x.std(ddof=1) if len(x) > 1 else np.nan, len(x)


def table(df, title, rows, cols, note=""):
    """rows: list of (label, run, test). Returns (markdown, latex, dataframe)."""
    recs = []
    for label, run, test in rows:
        r = {"Condition": label}
        n = 0
        for k in cols:
            m, s, n = cell(df, run, test, k)
            r[HDR.get(k, k)] = fmt(m, s, k)
            r["_" + k] = m
        r["n seeds"] = n
        recs.append(r)
    t = pd.DataFrame(recs)
    show = t[[c for c in t.columns if not c.startswith("_")]]
    show = show[show["n seeds"] > 0]
    md = f"### {title}\n\n" + (show.to_markdown(index=False) if len(show) else "_no runs yet_") + (f"\n\n{note}\n" if note else "\n")
    tex = ""
    if len(show):
        tex = f"% {title}\n\\begin{{tabular}}{{l{'c' * (len(show.columns) - 1)}}}\n\\hline\n"
        tex += " & ".join(show.columns).replace("±", "$\\pm$").replace("↓", "$\\downarrow$").replace("↑", "$\\uparrow$") + " \\\\\n\\hline\n"
        for _, r in show.iterrows():
            tex += " & ".join(str(v) for v in r.values).replace("±", "$\\pm$") + " \\\\\n"
        tex += "\\hline\n\\end{tabular}\n\n"
    t.attrs['show'] = show
    return md, tex, t


def paired(df, a, b, metric="mae", label=""):
    """a, b = (run, test). Pairs by seed."""
    A = df[(df.run == a[0]) & (df.test == a[1])].set_index("seed")[metric]
    B = df[(df.run == b[0]) & (df.test == b[1])].set_index("seed")[metric]
    s = A.index.intersection(B.index)
    if len(s) < 2:
        return None
    d = (A[s] - B[s]).astype(float).values
    t, p = stats.ttest_rel(A[s].values, B[s].values)
    ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=stats.sem(d)) if d.std() > 0 else (d.mean(), d.mean())
    return dict(comparison=label, metric=metric, A=f"{a[0]} @ {a[1]}", B=f"{b[0]} @ {b[1]}", n=len(s),
                mean_A=A[s].mean(), mean_B=B[s].mean(), diff=d.mean(), ci_low=ci[0], ci_high=ci[1], t=t, p=p,
                A_better_in=int((d < 0).sum() if metric == "mae" else (d > 0).sum()))


PUBLISHED = [  # Table 3 upper block: values as reported in the original papers (complete, aligned inputs)
    ("TFN [27]", "0.901", "0.698", "34.9", "73.9", "73.4"), ("LMF [28]", "0.917", "0.695", "33.2", "82.5", "82.4"),
    ("ICCN [29]", "0.862", "0.714", "39.0", "83.0", "83.0"), ("MulT [6]", "0.861", "0.711", "—", "84.1", "83.9"),
    ("MISA [7]", "0.804", "0.764", "—", "82.1", "82.0"), ("Self-MM [9]", "0.713", "0.798", "—", "85.98", "85.95"),
    ("MMIM [8]", "0.700", "0.800", "46.65", "86.06", "85.98"), ("JTUM [30]", "0.721", "0.798", "44.90", "86.28", "86.19"),
    ("DPM [32]", "0.668", "0.835", "45.34", "87.65", "87.61")]


class _A:
    pass


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--out", required=True); a = ap.parse_args()
    run(a.out)


def run(out_dir: str):
    a = _A(); a.out = out_dir
    if not os.path.exists(os.path.join(a.out, "per_run.csv")):
        print("analyze: no per_run.csv yet"); return
    df = pd.read_csv(os.path.join(a.out, "per_run.csv"))
    tdir = os.path.join(a.out, "tables"); os.makedirs(tdir, exist_ok=True)
    TABLES = {}
    info = json.load(open(os.path.join(a.out, "data_info.json"))) if os.path.exists(os.path.join(a.out, "data_info.json")) else {}
    MD, TEX = ["# RMSA-Net results (generated by analyze.py)\n"], []

    def add(md, tex, t=None, key=None):
        MD.append(md); TEX.append(tex)
        if t is not None and key:
            TABLES[key] = t
            t.attrs["show"].to_csv(os.path.join(tdir, key + ".csv"), index=False)

    # -------- data facts (R2.3, R1.7)
    MD.append("## Data facts\n")
    for k, v in info.items():
        MD.append(f"- **{k}**: constant mean-label predictor MAE = {v['const_mean_mae']:.3f}; predicting 0: {v['const_zero_mae']:.3f}. "
                  f"Train/val/test sizes and lengths: {json.dumps(v.get('stats', {}))}\n")
    pp = df.drop_duplicates("run")[["run", "n_params"]]
    MD.append("\n**Trainable parameters:** " + "; ".join(f"{r.run}: {r.n_params / 1e6:.2f}M" for r in pp.itertuples() if "rmsa-PCM|tr0-0" in r.run or "|tfn|" in r.run or "|mult|" in r.run) + "\n")

    # -------- Table 2 hyperparameters (taken from the code itself, so the paper cannot drift from it)
    import rmsa_net as RN
    d0 = next(iter(info.values()), {}).get("stats", {}).get("train", {}).get("d", ["?", "?", "?"])
    hp = [("Input dims (text, audio, vision)", ", ".join(map(str, d0))), ("BiLSTM hidden H (text: per direction; audio/vision: H/2 per direction)", "128"),
          ("BiLSTM layers (text / audio / vision)", "2 / 1 / 1"), ("Projection dim D", "64"), ("CMTA attention dim / heads", "128 / 1"),
          ("PSG gate and polarity MLP hidden", "128"), ("MMR surrogate generator", "64 → 128 → 64"), ("Fusion MLP", "192 → 128 → 32"),
          ("Dropout (text LSTM, fusion)", "0.3"), ("Modality dropout (training)", "per condition, = test rate (Table 3); 0 for clean"),
          ("Optimizer / LR / weight decay", "AdamW / 1e-3 / 1e-4"), ("LR schedule", "cosine annealing, η_min = 0, T_max = epochs"),
          ("Epochs / batch size / grad clip", f"{int(df.epochs.max()) if 'epochs' in df else 30} / 32 / 1.0"),
          ("Loss weights λ1 (BCE) / λ2 (unimodal) / λ3 (MMR-rec)", "0.5 / 0.1 / 0 (0.1 for MMR-rec)"),
          ("Model selection", "lowest validation MAE under the training condition"), ("Seeds", ", ".join(map(str, sorted(df.seed.unique())))),
          ("Feature standardisation", "audio/vision z-scored with training statistics"),
          ("Trainable parameters (full model)", f"{df[df.run.str.contains('rmsa-PCM')].n_params.iloc[0] / 1e6:.2f} M" if (df.run.str.contains('rmsa-PCM')).any() else "?")]
    t2 = pd.DataFrame(hp, columns=["Hyperparameter", "Value"]); t2.to_csv(os.path.join(tdir, "table2.csv"), index=False)
    MD.append("### Table 2 — Implementation hyperparameters\n\n" + t2.to_markdown(index=False) + "\n")
    TEX.append("% Table 2\n\\begin{tabular}{ll}\n\\hline\n" + "\n".join(f"{h} & {v} \\\\" for h, v in hp) + "\n\\hline\n\\end{tabular}\n\n")

    R = lambda ds, flags, tr, rec="": f"{ds}|rmsa-{flags}{rec}|{tr}"
    T0, T3, T5 = "tr0-0", "tr0.3-0.3", "tr0.5-0.5"

    # -------- Table 3 (matched train/test)
    rows = [("All modalities (clean)", R("mosi", "PCM", T0), "clean"),
            ("Unaligned (CMTA)", R("mosi_unal", "PCM", T0), "clean")]
    for lab, tr, te in [("Audio 30%", "tr0.3-0", "A30"), ("Vision 30%", "tr0-0.3", "V30"), ("A+V 30%", T3, "AV30"), ("A+V 50%", T5, "AV50")]:
        rows.append((f"{lab} + MMR", R("mosi", "PCM", tr), te))
    for lab, tr, te in [("Audio 30%", "tr0.3-0", "A30"), ("Vision 30%", "tr0-0.3", "V30"), ("A+V 30%", T3, "AV30"), ("A+V 50%", T5, "AV50")]:
        rows.append((f"{lab}, zero-fill", R("mosi", "PCZ", tr), te))
    md, tex, _t = table(df, "Table 3 — RMSA-Net on CMU-MOSI (training and test missingness matched)", rows, M5 + ["acc2_nz", "f1_nz"],
                       "Each missing-modality row is trained and tested with the stated rates; the clean row is trained without modality dropout.")
    add(md, tex, _t, "table3")
    pub = pd.DataFrame(PUBLISHED, columns=["Method", "MAE↓", "Corr↑", "Acc-7↑", "Acc-2↑", "F1↑"])
    pub.to_csv(os.path.join(tdir, "table3_published.csv"), index=False)
    MD.append("#### Table 3 (upper block) — prior methods as reported in their papers (complete, aligned inputs; different encoders; reference only)\n\n" + pub.to_markdown(index=False) + "\n")
    TEX.append("% Table 3 upper block (published)\n" + "\n".join(" & ".join(r) + " \\\\" for r in PUBLISHED) + "\n\n")

    # -------- Table 3b baselines
    rows = []
    for name, key in [("LF-LSTM", "lflstm"), ("TFN", "tfn"), ("MulT (re-impl.)", "mult")]:
        for lab, tr, te in [("clean", T0, "clean"), ("A+V 30%", T3, "AV30"), ("A+V 50%", T5, "AV50")]:
            rows.append((f"{name} — {lab}", f"mosi|{key}|{tr}", te))
        rows.append((f"{name} — unaligned", f"mosi_unal|{key}|{T0}", "clean"))
        rows.append((f"{name} — unaligned + A+V 30%", f"mosi_unal|{key}|{T3}", "AV30"))
    for lab, tr, te in [("clean", T0, "clean"), ("A+V 30%", T3, "AV30"), ("A+V 50%", T5, "AV50")]:
        rows.append((f"RMSA-Net zero-fill — {lab}", R("mosi", "PCZ", tr), te))
        rows.append((f"RMSA-Net MMR — {lab}", R("mosi", "PCM", tr), te))
    rows += [("RMSA-Net — unaligned", R("mosi_unal", "PCM", T0), "clean"), ("RMSA-Net — unaligned + A+V 30%", R("mosi_unal", "PCM", T3), "AV30")]
    md, tex, _t = table(df, "Table 3b — Baselines re-trained with identical features, seeds and degradation protocols", rows, M5,
                       "Baselines use zero-fill for missing streams, as they have no imputation mechanism.")
    add(md, tex, _t, "table3b")

    # -------- Table 3c: official TFR-Net (MMSA) under frame-level missingness vs RMSA-Net under the same test protocol
    tj = sorted(glob.glob(os.path.join(a.out, "tfrnet", "rate*_s*.json")))
    if tj:
        tf = pd.DataFrame([json.load(open(f)) for f in tj])
        recs = []
        for r, g in tf.groupby("rate"):
            recs.append({"Condition": f"TFR-Net (official MMSA, BERT fine-tuned), frame missing {r:g} in train+test",
                         **{HDR[k]: fmt(g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else np.nan, k) for k in M5}, "n seeds": len(g)})
        for lab, run, te in [("RMSA-Net MMR, clean-trained", R("mosi_unal", "PCM", T0), "clean"),
                             ("RMSA-Net MMR, clean-trained, frame missing 0.3 at test", R("mosi_unal", "PCM", T0), "frame30"),
                             ("RMSA-Net MMR, clean-trained, frame missing 0.5 at test", R("mosi_unal", "PCM", T0), "frame50"),
                             ("RMSA-Net MMR-rec, clean-trained, frame missing 0.3 at test", R("mosi_unal", "PCM", T0, "|rec0.1"), "frame30"),
                             ("RMSA-Net MMR-rec, clean-trained, frame missing 0.5 at test", R("mosi_unal", "PCM", T0, "|rec0.1"), "frame50"),
                             ("RMSA-Net MMR, A+V30-trained, frame missing 0.3 at test", R("mosi_unal", "PCM", T3), "frame30"),
                             ("RMSA-Net MMR, A+V30-trained, frame missing 0.5 at test", R("mosi_unal", "PCM", T3), "frame50")]:
            r0 = {"Condition": lab}
            n = 0
            for k in M5:
                m, s_, n = cell(df, run, te, k); r0[HDR[k]] = fmt(m, s_, k)
            r0["n seeds"] = n
            if n:
                recs.append(r0)
        t3c = pd.DataFrame(recs); t3c.to_csv(os.path.join(tdir, "table3c.csv"), index=False)
        MD.append("### Table 3c — Official TFR-Net vs RMSA-Net under frame-level missingness (unaligned CMU-MOSI)\n\n" + t3c.to_markdown(index=False)
                  + "\n\nTFR-Net is trained and tested with frame-level missingness (its own protocol) and fine-tunes BERT; "
                    "RMSA-Net uses frozen BERT features and is degraded only at test time. Not a like-for-like training regime.\n")
        globals()["_TFR"] = tf
    else:
        MD.append("### Table 3c — TFR-Net\n\n_not run (see run_tfrnet.py)_\n")

    # -------- Table 4 ablation
    rows = []
    for psg in "-P":
        for cm in "-C":
            for mm in "ZM":
                lab = " + ".join([x for x, on in [("PSG", psg == "P"), ("CMTA", cm == "C"), ("MMR", mm == "M")] if on]) or "Base BiLSTM"
                rows.append((lab, R("mosi", psg + cm + mm, T3), "AV30"))
    rows.sort(key=lambda r: (r[0] != "Base BiLSTM", r[0].count("+"), r[0]))
    md, tex, _t = table(df, "Table 4 — Factorial ablation, CMU-MOSI, A+V 30% (train and test)", rows, M5)
    add(md, tex, _t, "table4")
    abl = _t

    # -------- Table 5 MOSEI
    rows = [("All modalities (clean)", R("mosei", "PCM", T0), "clean"), ("A+V 30% + MMR", R("mosei", "PCM", T3), "AV30"),
            ("A+V 30%, zero-fill", R("mosei", "PCZ", T3), "AV30")]
    md, tex, _t = table(df, "Table 5 — CMU-MOSEI", rows, M5); add(md, tex, _t, "table5")

    # -------- Table 6 joint
    rows = [("Clean (ref.)", R("mosi", "PCM", T0), "clean"), ("A+V 30% only (ref.)", R("mosi", "PCM", T3), "AV30"),
            ("Unaligned only (ref.)", R("mosi_unal", "PCM", T0), "clean"), ("A+V 30% + unaligned, MMR", R("mosi_unal", "PCM", T3), "AV30"),
            ("A+V 30% + unaligned, zero-fill", R("mosi_unal", "PCZ", T3), "AV30")]
    md, tex, _t = table(df, "Table 6 — Joint degradation (CMU-MOSI)", rows, M5); add(md, tex, _t, "table6")

    # -------- Table 7 cross-evaluation (R1.8, R2.6)
    rows = []
    for mm, lab in [("Z", "Zero-fill"), ("M", "MMR")]:
        for tr, trl in [(T0, "0"), (T3, "0.3")]:
            for te, tel in [("clean", "0"), ("AV30", "0.3")]:
                rows.append((f"{lab} | p_train={trl} | p_test={tel}", R("mosi", "PC" + mm, tr), te))
    rows.append(("MMR-rec (λ3=0.1) | p_train=0 | p_test=0.3", R("mosi", "PCM", T0, "|rec0.1"), "AV30"))
    md, tex, _t = table(df, "Table 7 — Training-time dropout vs. test-time missingness (A+V)", rows, M5,
                       "Read (MMR, 0.3, 0) vs (MMR, 0, 0): effect of training with dropout on clean input. "
                       "(MMR, 0, 0.3) vs (Zero-fill, 0, 0.3): value of the surrogate for a model not trained for missing input.")
    add(md, tex, _t, "table7")

    # -------- Table 8 surrogate (R1.5, R2.7)
    rows = [("Zero-fill", R("mosi", "PCZ", T3), "AV30"), ("Text-MLP control (always on)", R("mosi", "PCX", T3), "AV30"),
            ("MMR-task (λ3=0)", R("mosi", "PCM", T3), "AV30"), ("MMR-rec (λ3=0.1)", R("mosi", "PCM", T3, "|rec0.1"), "AV30"),
            ("MMR-rec, trained clean (λ3=0.1)", R("mosi", "PCM", T0, "|rec0.1"), "AV30")]
    md, tex, _t = table(df, "Table 8 — What the MMR surrogate encodes (A+V 30% test)", rows,
                       ["fid_cos_a", "fid_cos_v", "fid_mse_a", "fid_mse_v", "mae", "acc7", "acc2"],
                       "Fidelity is computed on test samples where the modality is present (f̂m = Rm(ft) vs the true fm). "
                       "For zero-fill and text-MLP, Rm is untrained or not used as a substitute, so their fidelity is a chance-level reference.")
    add(md, tex, _t, "table8")

    # -------- Robustness to partial / noisy degradation (R2.7)
    rows = []
    for lab, run in [("Clean-trained MMR", R("mosi", "PCM", T0)), ("A+V30-trained MMR", R("mosi", "PCM", T3)),
                     ("A+V30-trained zero-fill", R("mosi", "PCZ", T3)), ("A+V30-trained MMR-rec", R("mosi", "PCM", T3, "|rec0.1"))]:
        for te in ["clean", "frame30", "frame50", "noise0.5", "noise1.0"]:
            rows.append((f"{lab} @ {te}", run, te))
    md, tex, _t = table(df, "Table 9 — Partial frame loss and additive noise (switch in Eq. 4 does not fire)", rows, M5); add(md, tex, _t, "table9")

    # -------- CMTA analysis (R1.7, R2.9)
    rows = [("Aligned, CMTA", R("mosi", "PCM", T0), "clean"), ("Aligned, no CMTA (mean-pool)", R("mosi", "P-M", T0), "clean"),
            ("Unaligned, CMTA", R("mosi_unal", "PCM", T0), "clean"), ("Unaligned, no CMTA (mean-pool)", R("mosi_unal", "P-M", T0), "clean")]
    md, tex, _t = table(df, "Table 10 — CMTA on aligned vs. unaligned input, with attention entropy (1 = uniform)", rows,
                       M5 + ["attn_entropy_a", "attn_entropy_v"]); add(md, tex, _t, "table10")

    # -------- paired tests
    C = [
        ("MMR vs zero-fill, A+V30", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "PCZ", T3), "AV30")),
        ("MMR vs zero-fill, A+V50", (R("mosi", "PCM", T5), "AV50"), (R("mosi", "PCZ", T5), "AV50")),
        ("MMR vs zero-fill, Audio30", (R("mosi", "PCM", "tr0.3-0"), "A30"), (R("mosi", "PCZ", "tr0.3-0"), "A30")),
        ("MMR vs zero-fill, Vision30", (R("mosi", "PCM", "tr0-0.3"), "V30"), (R("mosi", "PCZ", "tr0-0.3"), "V30")),
        ("A+V30 MMR vs clean (matched)", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "PCM", T0), "clean")),
        ("Dropout-trained vs clean-trained, clean test", (R("mosi", "PCM", T3), "clean"), (R("mosi", "PCM", T0), "clean")),
        ("Clean-trained: MMR vs zero-fill on A+V30 test", (R("mosi", "PCM", T0), "AV30"), (R("mosi", "PCZ", T0), "AV30")),
        ("MMR-rec vs MMR-task, A+V30", (R("mosi", "PCM", T3, "|rec0.1"), "AV30"), (R("mosi", "PCM", T3), "AV30")),
        ("MMR vs text-MLP control, A+V30", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "PCX", T3), "AV30")),
        ("Full vs base (ablation)", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "--Z", T3), "AV30")),
        ("PSG: full vs CMTA+MMR", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "-CM", T3), "AV30")),
        ("CMTA: full vs PSG+MMR", (R("mosi", "PCM", T3), "AV30"), (R("mosi", "P-M", T3), "AV30")),
        ("Unaligned CMTA vs aligned (clean)", (R("mosi_unal", "PCM", T0), "clean"), (R("mosi", "PCM", T0), "clean")),
        ("Unaligned: CMTA vs mean-pool", (R("mosi_unal", "PCM", T0), "clean"), (R("mosi_unal", "P-M", T0), "clean")),
        ("RMSA MMR vs MulT, A+V30", (R("mosi", "PCM", T3), "AV30"), ("mosi|mult|" + T3, "AV30")),
        ("RMSA MMR vs TFN, A+V30", (R("mosi", "PCM", T3), "AV30"), ("mosi|tfn|" + T3, "AV30")),
    ]
    st = []
    for lab, A, B in C:
        for m in ["mae", "acc7", "acc2"]:
            r = paired(df, A, B, m, lab)
            if r:
                st.append(r)
    st = pd.DataFrame(st)
    globals()['_ST'] = st
    if len(st):
        st.to_csv(os.path.join(a.out, "stats.csv"), index=False)
        show = st[["comparison", "metric", "n", "mean_A", "mean_B", "diff", "ci_low", "ci_high", "p", "A_better_in"]].copy()
        for c in ["mean_A", "mean_B", "diff", "ci_low", "ci_high"]:
            show[c] = show[c].map(lambda v: f"{v:.3f}")
        show["p"] = show["p"].map(lambda v: f"{v:.4f}")
        MD.append("### Paired comparisons (two-sided paired t-test over seeds; diff = A − B; 95% CI)\n\n" + show.to_markdown(index=False) + "\n")

    with open(os.path.join(a.out, "tables.md"), "w") as f:
        f.write("\n".join(MD))
    with open(os.path.join(a.out, "tables.tex"), "w") as f:
        f.write("\n".join(TEX))

    figures(df, a.out, TABLES.get('table4', pd.DataFrame({'n seeds': [0]})))
    psg_analysis(df, a.out)
    numbers_for_text(df, info, a.out, st)
    write_index(a.out)
    print(f"analyze: wrote RESULTS_INDEX.md, tables.md/.tex, tables/*.csv, stats.csv, numbers_for_text.md, figures and psg_analysis.md to {a.out}")


def figures(df, out, abl):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    # Figure 5: ablation
    if len(abl) and abl["n seeds"].max() > 0:
        lab = abl["Condition"].tolist()
        fig, ax = plt.subplots(figsize=(9, 3.6))
        x = np.arange(len(lab)); w = 0.27
        # recompute directly from the rows used in the table
        rows = []
        for psg in "-P":
            for cm in "-C":
                for mm in "ZM":
                    name = " + ".join([x for x, on in [("PSG", psg == "P"), ("CMTA", cm == "C"), ("MMR", mm == "M")] if on]) or "Base BiLSTM"
                    rows.append((name, f"mosi|rmsa-{psg}{cm}{mm}|tr0.3-0.3"))
        rows.sort(key=lambda r: (r[0] != "Base BiLSTM", r[0].count("+"), r[0]))
        for j, (k, c) in enumerate([("acc2", "#4C72B0"), ("f1", "#55A868"), ("acc7", "#C44E52")]):
            st = [cell(df, run, "AV30", k) for _, run in rows]
            ax.bar(x + (j - 1) * w, [s[0] for s in st], w, yerr=[0 if np.isnan(s[1]) else s[1] for s in st], label=HDR[k], color=c, capsize=2)
        ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=25, ha="right", fontsize=8)
        ax.set_ylabel("%"); ax.set_ylim(bottom=25); ax.legend(fontsize=8, ncol=3); ax.set_title("Factorial ablation, A+V 30% (mean ± std over seeds)", fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(out, "fig5_ablation.png"), dpi=200); plt.close(fig)
    # Figure 3: missing conditions, MMR vs zero-fill
    conds = [("Clean", "tr0-0", "clean"), ("Audio 30%", "tr0.3-0", "A30"), ("Vision 30%", "tr0-0.3", "V30"), ("A+V 30%", "tr0.3-0.3", "AV30"), ("A+V 50%", "tr0.5-0.5", "AV50")]
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
    for ax, k in zip(axs, ["mae", "acc2"]):
        for j, (mm, lab, c) in enumerate([("PCM", "MMR", "#4C72B0"), ("PCZ", "Zero-fill", "#DD8452")]):
            st = [cell(df, f"mosi|rmsa-{mm}|{tr}", te, k) for _, tr, te in conds]
            ax.bar(np.arange(len(conds)) + (j - 0.5) * 0.38, [s[0] for s in st], 0.38, yerr=[0 if np.isnan(s[1]) else s[1] for s in st], label=lab, color=c, capsize=2)
        ax.set_xticks(range(len(conds))); ax.set_xticklabels([c[0] for c in conds], fontsize=8); ax.set_title(HDR[k], fontsize=9)
        vals = [v for v in [cell(df, f"mosi|rmsa-{mm}|{tr}", te, k)[0] for mm in ("PCM", "PCZ") for _, tr, te in conds] if not np.isnan(v)]
        if vals:
            ax.set_ylim(min(vals) * 0.95, max(vals) * 1.03)
    axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(out, "fig3_missing.png"), dpi=200); plt.close(fig)
    # Figure 4: convergence (A+V30 full model), mean ± std over seeds
    cs = sorted(glob.glob(os.path.join(out, "curve_mosi_rmsa-PCM_tr0.3-0.3_s*.json")))
    if cs:
        C = np.array([json.load(open(f)) for f in cs])                  # (S, E, 3)
        ep = C[0, :, 0]
        fig, ax = plt.subplots(figsize=(6, 3.2)); ax2 = ax.twinx()
        for arr, axx, lab, col in [(C[:, :, 1], ax, "Validation MAE", "#4C72B0"), (C[:, :, 2], ax2, "Validation Acc-2", "#C44E52")]:
            m, s = arr.mean(0), arr.std(0)
            axx.plot(ep, m, color=col, label=lab); axx.fill_between(ep, m - s, m + s, color=col, alpha=0.2)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Validation MAE"); ax2.set_ylabel("Validation Acc-2 (%)")
        ax.set_title(f"Training convergence, A+V 30% (mean ± std, {len(cs)} seeds)", fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(out, "fig4_convergence.png"), dpi=200); plt.close(fig)
    # Figure 2: learned CMTA attention (unaligned model if available)
    npz = sorted(glob.glob(os.path.join(out, "examples_mosi_unal_rmsa-PCM_tr0-0_s*_attn.npz"))) or sorted(glob.glob(os.path.join(out, "examples_*PCM*_attn.npz")))
    if npz:
        z = np.load(npz[0])
        if z["A_ta"].size:
            i = 0; T, Ta, Tv = int(z["tl"][i]), int(z["al"][i]), int(z["vl"][i])
            fig, axs = plt.subplots(1, 2, figsize=(9, 3))
            for ax, A, L, lab in [(axs[0], z["A_ta"][i][:T, :Ta], Ta, "text → audio"), (axs[1], z["A_tv"][i][:T, :Tv], Tv, "text → vision")]:
                im = ax.imshow(A, aspect="auto", cmap="viridis"); ax.set_xlabel(f"{lab.split('→')[1].strip()} frame (native rate)"); ax.set_ylabel("text token"); ax.set_title(lab, fontsize=9)
                fig.colorbar(im, ax=ax, fraction=0.04)
            fig.suptitle(f"Learned CMTA attention, test segment index {int(z['index'][i])}", fontsize=9)
            fig.tight_layout(); fig.savefig(os.path.join(out, "fig2_attention.png"), dpi=200); plt.close(fig)


def bert_tokens_to_words(ex, L):
    """If the export carries BERT token ids, rebuild words exactly: drop [CLS]/[SEP]/[PAD], merge '##' word pieces.
    Word weight = sum of its pieces' alpha (attention mass on the word); |p| = max over pieces."""
    if not ex or not ex[0].get("token_ids"):
        return ex
    try:
        from transformers import BertTokenizer
        tok = BertTokenizer.from_pretrained("bert-base-uncased")
    except Exception as e:
        L.append(f"_Could not load the BERT tokenizer ({e}); install transformers to map tokens to words._\n")
        return ex
    out = []
    for e in ex:
        if e["alpha"] is None:
            out.append(e); continue
        toks = tok.convert_ids_to_tokens(e["token_ids"])
        words, al, ap = [], [], []
        for t, a, p in zip(toks, e["alpha"], e["abs_p"]):
            if t in ("[CLS]", "[SEP]", "[PAD]"):
                continue
            if t.startswith("##") and words:
                words[-1] += t[2:]; al[-1] += a; ap[-1] = max(ap[-1], p)
            else:
                words.append(t); al.append(a); ap.append(p)
        tot = sum(al) or 1.0
        out.append(dict(e, words=words, alpha=[x / tot for x in al], abs_p=ap,
                        special_mass=1.0 - sum(al)))
    sm = np.mean([e.get("special_mass", 0) for e in out])
    L.append(f"Token-to-word mapping uses the BERT token ids stored in the data file. On average {sm * 100:.1f}% of PSG weight "
             f"falls on [CLS]/[SEP] tokens; word weights below are renormalised over real words.\n")
    return out


def psg_analysis(df, out):
    L = ["# PSG analysis (R1.6, R2.8)\n"]
    if "psg_signflip_maxdiff" in df:
        v = df["psg_signflip_maxdiff"].dropna()
        if len(v):
            L.append(f"**Sign-flip check.** Negating the last layer of the polarity estimator changed the regression output by at most "
                     f"{v.max():.2e} over {len(v)} trained models (float rounding only). The sign of p is therefore not identifiable; only |p| is meaningful.\n")
    js = sorted(glob.glob(os.path.join(out, "examples_mosi_rmsa-PCM_tr0.3-0.3_s*_psg.json"))) or sorted(glob.glob(os.path.join(out, "examples_mosi_*_psg.json")))
    if not js:
        open(os.path.join(out, "psg_analysis.md"), "w").write("\n".join(L)); return
    ex = json.load(open(js[0]))
    ex = bert_tokens_to_words(ex, L)
    if ex and ex[0]["words"] is not None:
        try:
            from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
            lex = SentimentIntensityAnalyzer().lexicon
            xs, ys, used = [], [], 0
            for e in ex:
                if e["abs_p"] is None or e["words"] is None or len(e["words"]) != len(e["abs_p"]):
                    continue
                used += 1
                for w, p in zip(e["words"], e["abs_p"]):
                    w = w.lower().strip(".,!?\"'")
                    if w in lex:
                        xs.append(p); ys.append(abs(lex[w]))
            if len(xs) > 10:
                rho, pv = stats.spearmanr(xs, ys)
                L.append(f"**|p| vs. VADER lexicon intensity.** Spearman ρ = {rho:.3f} (p = {pv:.2g}) over {len(xs)} lexicon tokens in {used} "
                         f"test utterances whose word count matches the token count. The lexicon is used only for this analysis.\n")
        except ImportError:
            L.append("_Install vaderSentiment (`pip install vaderSentiment`) to compute the lexicon correlation._\n")
        L.append("**Qualitative examples (top-3 tokens by PSG weight).**\n")
        exs = sorted([e for e in ex if e["words"] and e["alpha"] and len(e["words"]) == len(e["alpha"])], key=lambda e: -abs(e["label"]))[:4]
        exs += sorted([e for e in ex if e["words"] and e["alpha"] and len(e["words"]) == len(e["alpha"])], key=lambda e: -abs(e["label"] - e["pred"]))[:2]
        for e in exs:
            top = np.argsort(e["alpha"])[::-1][:3]
            L.append(f"- label {e['label']:+.1f}, prediction {e['pred']:+.2f}: “{' '.join(e['words'])}” → top tokens: "
                     + ", ".join(f"{e['words'][i]} ({e['alpha'][i]:.2f})" for i in top) + "\n")
        try:
            import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
            sel = exs[:4]
            if sel:
                fig, axs = plt.subplots(len(sel), 1, figsize=(10, 1.5 * len(sel)))
                axs = np.atleast_1d(axs)
                for ax, e in zip(axs, sel):
                    ax.bar(range(len(e["alpha"])), e["alpha"], color="#4C72B0")
                    ax.set_xticks(range(len(e["words"]))); ax.set_xticklabels(e["words"], rotation=60, fontsize=7)
                    ax.set_ylabel("α", fontsize=8); ax.set_title(f"label {e['label']:+.1f}, prediction {e['pred']:+.2f}", fontsize=8)
                fig.tight_layout(); fig.savefig(os.path.join(out, "fig6_psg.png"), dpi=200); plt.close(fig)
        except Exception as ex:
            L.append(f"_Figure 6 not drawn: {ex}_\n")
    else:
        L.append("_The data file has no raw_text, so tokens cannot be mapped to words. Use the MMSA release (contains raw_text) for this analysis._\n")
    open(os.path.join(out, "psg_analysis.md"), "w").write("\n".join(L))


# =====================================================================
# Numbers quoted in the manuscript text (abstract, introduction, Sections V-VII)
# =====================================================================
def numbers_for_text(df, info, out, st):
    T0, T3, T5 = "tr0-0", "tr0.3-0.3", "tr0.5-0.5"
    R = lambda ds, flags, tr, rec="": f"{ds}|rmsa-{flags}{rec}|{tr}"
    g = lambda run, test, k: cell(df, run, test, k)[0]
    L = ["# Numbers for the manuscript text (generated)\n",
         "Every number below is computed from per_run.csv. Copy them into the text; do not type numbers by hand. "
         "p-values are two-sided paired t-tests over seeds (see stats.csv).\n"]

    def pv(label, metric="mae"):
        if st is None or not len(st):
            return ""
        r = st[(st.comparison == label) & (st.metric == metric)]
        return f" (paired p = {r.p.iloc[0]:.3g}, 95% CI [{r.ci_low.iloc[0]:.3f}, {r.ci_high.iloc[0]:.3f}], better in {int(r.A_better_in.iloc[0])}/{int(r.n.iloc[0])} seeds)" if len(r) else ""

    def line(where, text):
        L.append(f"- **{where}** — {text}")

    c, m3, z3, m5, z5 = R("mosi", "PCM", T0), R("mosi", "PCM", T3), R("mosi", "PCZ", T3), R("mosi", "PCM", T5), R("mosi", "PCZ", T5)
    L.append("\n## Headline (abstract E-02, introduction E-08)\n")
    line("Clean", f"MAE {g(c,'clean','mae'):.3f}, Corr {g(c,'clean','corr'):.3f}, Acc-7 {g(c,'clean','acc7'):.2f}%, Acc-2 {g(c,'clean','acc2'):.2f}%, F1 {g(c,'clean','f1'):.2f}%")
    line("A+V 30% + MMR", f"MAE {g(m3,'AV30','mae'):.3f}, Corr {g(m3,'AV30','corr'):.3f}, Acc-7 {g(m3,'AV30','acc7'):.2f}%, Acc-2 {g(m3,'AV30','acc2'):.2f}%, F1 {g(m3,'AV30','f1'):.2f}%")
    line("A+V 30% zero-fill", f"MAE {g(z3,'AV30','mae'):.3f}, Acc-2 {g(z3,'AV30','acc2'):.2f}%")
    line("MMR vs zero-fill (A+V 30%)", f"ΔMAE = {g(m3,'AV30','mae') - g(z3,'AV30','mae'):+.3f}" + pv("MMR vs zero-fill, A+V30"))
    line("MMR vs zero-fill (A+V 50%)", f"ΔMAE = {g(m5,'AV50','mae') - g(z5,'AV50','mae'):+.3f}" + pv("MMR vs zero-fill, A+V50"))
    tot = g(m3, 'AV30', 'mae') - g(c, 'clean', 'mae'); zf = g(z3, 'AV30', 'mae') - g(c, 'clean', 'mae'); mm = g(m3, 'AV30', 'mae') - g(z3, 'AV30', 'mae')
    line("Decomposition A+V30 vs clean", f"total ΔMAE {tot:+.3f} = zero-fill part {zf:+.3f} + MMR part {mm:+.3f}" + pv("A+V30 MMR vs clean (matched)"))

    L.append("\n## Training-time dropout vs test-time missingness (Table 7; E-41, E-44, E-60)\n")
    line("Clean test", f"dropout-trained MMR {g(m3,'clean','mae'):.3f} vs clean-trained {g(c,'clean','mae'):.3f} → Δ {g(m3,'clean','mae') - g(c,'clean','mae'):+.3f}" + pv("Dropout-trained vs clean-trained, clean test"))
    line("A+V30 test, clean-trained model", f"MMR {g(c,'AV30','mae'):.3f} vs zero-fill {g(R('mosi','PCZ',T0),'AV30','mae'):.3f}" + pv("Clean-trained: MMR vs zero-fill on A+V30 test")
         + f"; MMR-rec trained clean: {g(R('mosi','PCM',T0,'|rec0.1'),'AV30','mae'):.3f}")
    line("Cost of missing input at test", f"dropout-trained MMR: clean {g(m3,'clean','mae'):.3f} → A+V30 {g(m3,'AV30','mae'):.3f}")

    L.append("\n## Ablation (Table 4; E-53, E-54, abstract PSG gain)\n")
    full = m3
    for lab, other, cmp in [("PSG (full vs CMTA+MMR)", R("mosi", "-CM", T3), "PSG: full vs CMTA+MMR"), ("CMTA (full vs PSG+MMR)", R("mosi", "P-M", T3), "CMTA: full vs PSG+MMR"),
                            ("MMR (full vs PSG+CMTA zero-fill)", z3, "MMR vs zero-fill, A+V30"), ("All modules (full vs base)", R("mosi", "--Z", T3), "Full vs base (ablation)")]:
        line(lab, f"ΔAcc-7 {g(full,'AV30','acc7') - g(other,'AV30','acc7'):+.2f} pp, ΔMAE {g(full,'AV30','mae') - g(other,'AV30','mae'):+.3f}" + pv(cmp, "acc7"))

    L.append("\n## MMR surrogate (Table 8; E-23, E-47)\n")
    for lab, run in [("MMR-task", m3), ("MMR-rec", R("mosi", "PCM", T3, "|rec0.1")), ("Zero-fill (untrained R)", z3), ("Text-MLP control", R("mosi", "PCX", T3))]:
        line(lab, f"cos(f̂a,fa) {g(run,'AV30','fid_cos_a'):.3f}, cos(f̂v,fv) {g(run,'AV30','fid_cos_v'):.3f}, MAE {g(run,'AV30','mae'):.3f}")
    line("MMR vs text-MLP", "" + pv("MMR vs text-MLP control, A+V30"))
    line("MMR-rec vs MMR-task", "" + pv("MMR-rec vs MMR-task, A+V30"))

    L.append("\n## Unaligned / CMTA (Tables 3, 10; E-43, E-48)\n")
    u = R("mosi_unal", "PCM", T0)
    line("Unaligned vs aligned", f"Acc-7 {g(u,'clean','acc7'):.2f} vs {g(c,'clean','acc7'):.2f} (Δ {g(u,'clean','acc7') - g(c,'clean','acc7'):+.2f}); Acc-2 {g(u,'clean','acc2'):.2f} vs {g(c,'clean','acc2'):.2f}; MAE {g(u,'clean','mae'):.3f} vs {g(c,'clean','mae'):.3f}" + pv("Unaligned CMTA vs aligned (clean)", "acc7"))
    line("Unaligned: CMTA vs mean-pool", f"ΔAcc-7 {g(u,'clean','acc7') - g(R('mosi_unal','P-M',T0),'clean','acc7'):+.2f}" + pv("Unaligned: CMTA vs mean-pool", "acc7"))
    line("Attention entropy (1 = uniform)", f"audio {g(u,'clean','attn_entropy_a'):.3f}, vision {g(u,'clean','attn_entropy_v'):.3f}")
    j = R("mosi_unal", "PCM", T3)
    line("Joint degradation (Table 6)", f"MAE {g(j,'AV30','mae'):.3f}, Acc-7 {g(j,'AV30','acc7'):.2f}, Acc-2 {g(j,'AV30','acc2'):.2f}")

    L.append("\n## Baselines (Table 3b; E-39)\n")
    for k in ["lflstm", "tfn", "mult"]:
        line(k.upper(), f"clean MAE {g(f'mosi|{k}|{T0}','clean','mae'):.3f}; A+V30 MAE {g(f'mosi|{k}|{T3}','AV30','mae'):.3f}; A+V50 MAE {g(f'mosi|{k}|{T5}','AV50','mae'):.3f}")
    line("RMSA vs MulT (A+V30)", pv("RMSA MMR vs MulT, A+V30")); line("RMSA vs TFN (A+V30)", pv("RMSA MMR vs TFN, A+V30"))

    tf = globals().get("_TFR")
    if tf is not None:
        for r, g2 in tf.groupby("rate"):
            line(f"TFR-Net official, frame missing {r:g}", f"MAE {g2.mae.mean():.3f} ± {g2.mae.std(ddof=1):.3f}, Acc-7 {g2.acc7.mean():.2f}, Acc-2 {g2.acc2.mean():.2f} ({len(g2)} seeds)")
    L.append("\n## Robustness to partial / noisy input (Table 9; E-22, E-61)\n")
    for te in ["frame30", "noise0.5", "noise1.0"]:
        line(te, f"A+V30-trained MMR MAE {g(m3,te,'mae'):.3f} vs zero-fill {g(z3,te,'mae'):.3f} vs clean-trained {g(c,te,'mae'):.3f}")

    L.append("\n## MOSEI (Table 5)\n")
    line("MOSEI", f"clean MAE {g(R('mosei','PCM',T0),'clean','mae'):.3f}; A+V30 MMR {g(R('mosei','PCM',T3),'AV30','mae'):.3f}; zero-fill {g(R('mosei','PCZ',T3),'AV30','mae'):.3f}")

    L.append("\n## Protocol facts (Section IV; E-31, E-32, E-34, E-51)\n")
    for k, v in info.items():
        line(f"{k} constant predictor", f"MAE {v['const_mean_mae']:.3f} (predicting the training mean)")
        s = v.get("stats", {}).get("train", {})
        if s:
            line(f"{k} data", f"dims {s.get('d')}, max lengths T/Ta/Tv = {s.get('T_text')}/{s.get('T_audio')}/{s.get('T_vision')}, "
                              f"training segments with all-zero audio {s.get('audio_all_zero')} / vision {s.get('vision_all_zero')} of {s.get('n')}")
    if "n_params" in df:
        pr = df[df.run.str.contains("rmsa-PCM")]
        if len(pr):
            line("Parameters (full model)", f"{pr.n_params.iloc[0]:,}")
    be = df[(df.run == m3) & (df.test == "AV30")].best_epoch
    if len(be):
        line("Best epoch (A+V30 full)", f"range {be.min()}–{be.max()} over {len(be)} seeds")
    line("Compute", f"{df.drop_duplicates(['run','seed']).train_sec.sum() / 3600:.1f} GPU-hours for {df.drop_duplicates(['run','seed']).shape[0]} runs "
                    f"({df.run.nunique()} configurations × {df.seed.nunique()} seeds)")
    open(os.path.join(out, "numbers_for_text.md"), "w").write("\n".join(L).replace("nan", "—") + "\n")


def write_index(out):
    items = [("Table 2", "tables/table2.csv, tables.md", "hyperparameters (from the code)", "R2.1"),
             ("Table 3", "tables/table3.csv + table3_published.csv", "main results, MMR and zero-fill, matched train/test", "R1.1, R1.2, R2.1, R2.2"),
             ("Table 3b", "tables/table3b.csv", "baselines re-trained under identical protocols", "R1.4, R2.4"),
             ("Table 3c", "tables/table3c.csv", "official TFR-Net (MMSA) vs RMSA-Net under frame-level missingness", "R1.4, R2.4"),
             ("Table 4", "tables/table4.csv", "factorial ablation (A+V 30%)", "R1.1, R2.1"),
             ("Table 5", "tables/table5.csv", "CMU-MOSEI", "—"),
             ("Table 6", "tables/table6.csv", "joint degradation", "R2.1"),
             ("Table 7", "tables/table7.csv", "training dropout vs test missingness (2×2)", "R1.8, R2.6"),
             ("Table 8", "tables/table8.csv", "what the MMR surrogate encodes (fidelity, MMR-rec, text-MLP control)", "R1.5, R2.7"),
             ("Table 9", "tables/table9.csv", "partial frame loss and noise", "R2.7"),
             ("Table 10", "tables/table10.csv", "CMTA vs mean-pool, attention entropy", "R1.7, R2.9"),
             ("Figure 2", "fig2_attention.png", "learned CMTA attention (replaces schematic)", "R1.7, R2.9"),
             ("Figure 3", "fig3_missing.png", "missing-modality conditions, MMR vs zero-fill", "R1.1"),
             ("Figure 4", "fig4_convergence.png", "convergence, mean ± std over seeds", "R2.1"),
             ("Figure 5", "fig5_ablation.png", "ablation, same numbers as Table 4", "R1.1"),
             ("Figure 6", "fig6_psg.png", "PSG token weights, qualitative examples", "R1.6, R2.8"),
             ("Statistics", "stats.csv", "paired t-tests, 95% CI, seeds favouring A", "R1.2, R2.2"),
             ("Text numbers", "numbers_for_text.md", "every number quoted in abstract / introduction / Sections V–VII", "R1.1, R2.1"),
             ("PSG analysis", "psg_analysis.md", "sign-flip check, |p| vs lexicon, examples", "R1.6, R2.8"),
             ("LaTeX", "tables.tex", "all tables as tabular blocks", "—"),
             ("Raw", "per_run.csv, data_info.json, checkpoints/", "per-seed results (submit as supplementary)", "R1.2")]
    L = ["# Results index\n", "| Paper item | File | Content | Reviewer comments |", "|---|---|---|---|"]
    for a, f, c, r in items:
        ok = all(os.path.exists(os.path.join(out, x.strip().split(" ")[0])) for x in f.split(",") if x.strip() and not x.strip().endswith("/") and "+" not in x and "tables.md" not in x)
        L.append(f"| {a} | {f} | {c} | {r} |" + ("" if ok else " ⚠ not generated yet"))
    open(os.path.join(out, "RESULTS_INDEX.md"), "w").write("\n".join(L) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/rmsa_code/run_tfrnet.py
"""
TFR-Net baseline (R1.4, R2.4) with the official implementation in the MMSA toolkit, on the same CMU-MOSI
unaligned file and the same five seeds as RMSA-Net. Resume-safe: one JSON per (rate, seed) in OUT/tfrnet/.

TFR-Net's own missing-data protocol drops random FRAMES (not whole streams) at the given rate, in training and
test, and fine-tunes BERT. Text is kept complete (rate 0) so the setting matches RMSA-Net's audio/visual
degradation as closely as the official code allows; compare it with the frame30 rows of Table 9.

    pip install MMSA
    python run_tfrnet.py --unaligned /content/data/unaligned_50.pkl --out /content/drive/MyDrive/rmsa_results_v2
"""
import argparse, ast, json, os, time


def parse_cell(v):
    """MMSA writes '(mean*100, std*100)' per metric."""
    v = ast.literal_eval(v) if isinstance(v, str) else v
    return float(v[0]) if isinstance(v, (tuple, list)) else float(v)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--unaligned", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--seeds", nargs="+", type=int, default=[13, 42, 7, 17, 99])
    ap.add_argument("--rates", nargs="+", type=float, default=[0.0, 0.3, 0.5])
    a = ap.parse_args()
    import pandas as pd
    from MMSA import MMSA_run
    base = os.path.join(a.out, "tfrnet"); os.makedirs(base, exist_ok=True)
    for r in a.rates:
        for s in a.seeds:
            dst = os.path.join(base, f"rate{r:g}_s{s}.json")
            if os.path.exists(dst):
                print("done:", dst); continue
            raw = os.path.join(base, f"raw_rate{r:g}_s{s}")
            t0 = time.time()
            MMSA_run("tfr_net", "mosi", seeds=[s], gpu_ids=[0], num_workers=2,
                     config={"featurePath": a.unaligned, "missing_rate": [0.0, r, r], "missing_seed": [1111 + s] * 3},
                     res_save_dir=raw, model_save_dir="/content/mmsa_models", log_dir=raw)
            row = pd.read_csv(os.path.join(raw, "normal", "mosi.csv")).iloc[-1]
            res = dict(model="tfr_net", rate=r, seed=s, train_sec=round(time.time() - t0, 1),
                       mae=parse_cell(row["MAE"]) / 100, corr=parse_cell(row["Corr"]) / 100,
                       acc7=parse_cell(row["Mult_acc_7"]), acc2=parse_cell(row["Has0_acc_2"]),
                       f1=parse_cell(row["Has0_F1_score"]), acc2_nz=parse_cell(row["Non0_acc_2"]),
                       f1_nz=parse_cell(row["Non0_F1_score"]))
            json.dump(res, open(dst, "w"))
            print(res)


if __name__ == "__main__":
    main()


## 2. Data (CMU-MOSI shortcut in My Drive) and paths

In [ ]:
import glob, shutil
hits = {n: glob.glob(f'/content/drive/MyDrive/CMU-MOSI/**/{n}', recursive=True) for n in ['aligned_50.pkl', 'unaligned_50.pkl']}
for n, h in hits.items(): print(n, '->', h[0] if h else 'NOT FOUND')
os.makedirs('/content/data', exist_ok=True)
for n, h in hits.items():
    if h and not os.path.exists(f'/content/data/{n}'):
        shutil.copy(h[0], f'/content/data/{n}')
MOSI_ALIGNED, MOSI_UNALIGNED = '/content/data/aligned_50.pkl', '/content/data/unaligned_50.pkl'
OUT = '/content/drive/MyDrive/rmsa_results_v2'
%cd /content/rmsa_code

## 3. RMSA-Net and in-pipeline baselines (about 4–5 GPU-hours in total)

In [ ]:
!python rmsa_net.py --mosi_aligned $MOSI_ALIGNED --mosi_unaligned $MOSI_UNALIGNED --groups main ablation --out $OUT

In [ ]:
!python rmsa_net.py --mosi_aligned $MOSI_ALIGNED --mosi_unaligned $MOSI_UNALIGNED --groups mmr_analysis unaligned --out $OUT

In [ ]:
!python rmsa_net.py --mosi_aligned $MOSI_ALIGNED --mosi_unaligned $MOSI_UNALIGNED --groups baselines --out $OUT

## 4. Official TFR-Net (MMSA toolkit), 3 rates × 5 seeds
Fine-tunes BERT, so it is slower (roughly 5–15 minutes per run). Resume-safe: each finished (rate, seed) is saved as JSON.

In [ ]:
!pip -q install MMSA easydict nvidia-ml-py3
!python run_tfrnet.py --unaligned $MOSI_UNALIGNED --out $OUT

## 5. Build all results and download them

In [ ]:
!python rmsa_net.py --analyze_only --out $OUT
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/RESULTS_INDEX.md').read()))
!cd $OUT && zip -q -r /content/rmsa_results_v2.zip . -x 'checkpoints/*' 'curve_*' 'tfrnet/raw_*'
from google.colab import files; files.download('/content/rmsa_results_v2.zip')